<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_24_software_ecosystem_reproducibility.ipynb">9.24 软件生态与可复现实践</a></li>
        <li>下一节： <a href="9_26_low_high_frequency_special_regimes.ipynb">9.26 低频与高频特殊观测体制</a></li>
    </ul>
</div>


## 9.25 VLBI 实践入口：延迟、Fringe Fitting、相位参考与天体测量

极长基线干涉测量（Very Long Baseline Interferometry, VLBI）常被直观地理解为“把基线做得更长”。这种说法只说对了一半。长基线带来毫角秒甚至微角秒量级的角分辨率，但 VLBI 真正改变数据处理性质的地方，是阵元之间不再共享同一套本振、时钟、传输链路和实时相关环境。每个台站独立接收、独立记录，再依靠氢钟、精密时间标记、台站坐标、地球定向参数、传播介质模型和相关器延迟模型，把相距数百到数千公里的电压流重新放到同一个相干坐标系中。

前面章节已经建立了可见度、基线、RIME、校准和成像的基本语言。VLBI 不需要抛弃这些语言，而是把其中几个原本可以近似隐藏的量推到中心位置：几何延迟不再是一个小修正，台站钟差不再被共同本振抵消，大气和电离层误差会直接限制相干时间，相位校准不仅影响图像质量，还可能决定天体测量位置是否可信。因此，VLBI 的实践入口不是某个软件命令，而是三个问题：信号是否被正确对齐，残余相位能否在频率和时间上保持相干，校准解是否仍保留绝对位置和通量信息。

这一节作为实践章的 VLBI 入口，不试图替代完整的 VLBI 专门课程。目标是把连通阵列处理经验转化为 VLBI 可用的判断框架：fringe fitting 究竟在解什么，phase referencing 为什么能做天体测量，SEFD 和系统温度如何进入振幅标尺，稀疏长基线 $uv$ 覆盖为何使 VLBI 成像更依赖模型、闭合量和保守验证。


### 9.25.1 共同的可见度语言与关键差异

普通连通阵列和 VLBI 都测量天空亮度分布的相干信息。忽略宽场和偏振细节时，一条基线上的观测量仍可写成

$$
V_{pq}^{\rm obs}(t,\nu)=g_p(t,\nu)g_q^*(t,\nu)V_{pq}^{\rm true}(t,\nu)+n_{pq}(t,\nu),
$$

其中 $p,q$ 是两个台站或天线，$g_p$ 和 $g_q$ 是站点相关复增益，$V_{pq}^{\rm true}$ 包含天空结构的傅里叶分量，$n_{pq}$ 是热噪声。这个形式说明 VLBI 并不是另一套物理；它仍然属于相干干涉测量，仍然需要模型、权重、校准和反演。

关键差异在于 $g_p$ 的相位结构。连通阵列通常有共同频率标准和受控的信号传输链，许多快速变化项可以通过阵列内部系统稳定性和常规增益校准吸收。VLBI 的每个站点则带着独立时钟和独立接收链，站点相关相位常可写成近似形式

$$
g_p(t,\nu)=a_p(t,\nu)\exp\left[i\phi_p(t,\nu)
+i2\pi\nu\tau_p(t)+i2\pi f_p(t-t_0)\right],
$$

这里的 $\tau_p$ 表示站点相关残余延迟，$f_p$ 表示残余 fringe rate，$a_p$ 表示振幅标尺。单个站点相位无法直接观测，实际求解发生在基线或全阵的差分量上。只要某个台站的钟差、传播延迟或几何模型误差没有被充分消除，基线相位就会随频率和时间快速旋转，频带内相干平均和长时间积分都会损失信号。

因此，VLBI 的“高分辨率”伴随一个代价：信号本身往往更难检测。目标必须足够紧致，亮温足够高，且校准源足够近、足够亮、结构足够简单。对扩展低面亮度发射，VLBI 可能有极高角分辨率，却几乎没有可恢复通量；对紧致水脉泽、AGN 核、喷流、脉冲星和精密位置测量，VLBI 又能提供普通阵列很难达到的信息。


### 9.25.2 延迟模型、台站钟与 fringe stopping

一条基线的几何延迟可以从波前到达两个台站的时间差理解。若 $\mathbf{b}_{pq}=\mathbf{r}_q-\mathbf{r}_p$ 是基线向量，$\hat{\mathbf{s}}$ 是源方向，则几何延迟的量级为

$$
\tau_g\simeq \frac{\mathbf{b}_{pq}\cdot\hat{\mathbf{s}}}{c}.
$$

对洲际基线，$|\mathbf{b}|$ 可达 $10^6$ 到 $10^7\,\mathrm{m}$，几何延迟可以达到毫秒到数十毫秒量级。相关器必须在电压相关前把不同台站的数据流按模型延迟对齐，否则同一波前的电场样本不会相互重合。这个模型不仅需要台站位置和源坐标，还需要地球自转、极移、章动、岁差、相对论项、对流层、电离层和台站钟模型。

相关器完成的是先验对齐，而不是完美校准。模型误差会留下残余延迟和残余 fringe rate。对一个窄时间、窄频率窗口，残余相位常写成

$$
\phi_{pq}(t,\nu)\simeq \phi_0 +2\pi(\nu-\nu_0)\tau_{\rm res}
+2\pi(t-t_0)f_{\rm res}+\phi_{\rm str}(t,\nu),
$$

其中 $\tau_{\rm res}$ 是残余群延迟，$f_{\rm res}$ 是残余 fringe rate，$\phi_{\rm str}$ 是源结构造成的相位项。若 $\tau_{\rm res}$ 过大，频带两端相位差会很快超过 $1$ rad，频率平均将退相干；若 $f_{\rm res}$ 过大，时间平均同样会把信号平均掉。fringe stopping 的作用，是在相关或后处理阶段去掉可预报的快速相位旋转，把剩余问题压缩成可由 fringe fitting 求解的小量。

这个视角也解释了氢钟的重要性。VLBI 不是要求每个站点的钟绝对完美，而是要求钟在观测时间内足够稳定，使残余延迟和残余速率可以被模型和校准源约束。氢钟、GPS 时间传递、台站日志、相位校准信号、地球定向参数和气象信息共同决定了相关后数据是否具有可检测的 fringe。


![VLBI station clock delay model](figures/practical_vlbi_station_clock_delay_model.png)

图中把 VLBI 的相关前提压缩成一张示意图。每个台站都有独立时钟和独立接收链；相关器使用几何延迟、台站钟、地球定向参数、传播介质和源位置建立先验延迟模型。fringe fitting 面对的不是总延迟，而是先验模型之后留下的残余延迟、残余相位和残余 fringe rate。


### 9.25.3 Fringe fitting：从相位斜率到相干检测

fringe fitting 是 VLBI 数据处理的核心入口。它的基本思想可以直接从上一式读出：相位对频率的斜率给出残余群延迟，相位对时间的斜率给出残余 fringe rate。若把一个解区间内的 visibility 记为 $V(t_l,\nu_k)$，一个简化的 delay-rate 搜索可以写成

$$
S(\tau,f)=\left|\sum_{l,k} w_{lk}V(t_l,\nu_k)
\exp\{-i2\pi[(\nu_k-\nu_0)\tau+(t_l-t_0)f]\}\right|.
$$

当试探的 $\tau$ 和 $f$ 接近真实残余值时，不同时间和频率上的复数相位被转回同一方向，求和幅度达到峰值；试探值错误时，复数在相量平面上互相抵消。这个表达式说明 fringe fitting 不是神秘操作，而是在有限相干时间和有限带宽内寻找能最大化相干叠加的站点相关参数。

实际 fringe fitting 比这个公式更复杂。首先，延迟和 fringe rate 通常按站点相关量求解，而不是每条基线独立求解，因为站点解可以利用全阵约束并保持闭合关系。其次，亮而结构简单的 fringe finder 可用于求解仪器延迟和钟模型，较近的相位参考源用于跟踪时间变化，目标源只有在足够亮或经过相位参考后才适合自校准。再次，源结构会产生非零相位；若把有明显喷流结构的校准源当作点源，fringe 解可能吸收结构相位，随后把错误转移到目标场。

fringe fitting 的成功与否首先由信噪比决定。增加带宽和解区间能提高相干和的信噪比，但解区间不能长过大气、钟差和源结构可近似稳定的时间尺度。低频受电离层影响明显，毫米波受对流层相位快速变化影响明显；弱源、长基线、高 SEFD 台站和错误源模型都会降低 fringe detection 的可靠性。实践中，fringe S/N、delay-rate 峰是否孤立、不同基线和不同 IF 是否一致，比单次成像结果更早暴露问题。


![VLBI fringe fit delay rate](figures/practical_vlbi_fringe_fit_delay_rate.png)

左图示意残余延迟造成相位随频率倾斜，中图示意残余 fringe rate 造成相位随时间倾斜。右图把 delay-rate 搜索看成相干信噪比的二维峰值寻找。实际软件会加入站点相关约束、频带组合、源模型和质量标记，但核心物理仍是相位斜率的测量。


### 9.25.4 相位参考与天体测量

自校准可以显著改善 VLBI 图像，但它会牺牲一部分绝对相位信息。源位置误差在 visibility 中表现为随基线变化的相位斜坡；若直接对目标做自由相位自校准，这个相位斜坡可能被站点相位吸收，图像会变漂亮，但绝对位置会失去约束。相位参考（phase referencing）的目标，是用一个位置已知、足够近、足够亮、足够紧致的校准源跟踪相位扰动，再把解转移到目标源，使目标图像仍保留相对于校准源的位置信息。

一个典型相位参考观测在校准源和目标源之间反复切换。切换周期必须短于主要相位扰动的变化时间，角距离必须小到两条视线穿过的大气和电离层相似。残余路径误差 $\Delta L$ 会造成相位误差

$$
\Delta\phi = \frac{2\pi\Delta L}{\lambda},
$$

并进一步转化为目标位置偏差。热噪声给出的理想位置精度常近似为

$$
\sigma_\theta\sim \frac{\theta_{\rm beam}}{2\,\mathrm{SNR}},
$$

但高精度 VLBI 天体测量通常不是由这个热噪声极限决定，而是由系统误差决定：校准源和目标之间的角距离、对流层延迟残差、电离层色散延迟、校准源结构、频率相关 core shift、台站坐标误差和地球定向参数都会留下不可简单平均掉的偏差。

这也是 VLBI 观测设计和后期处理强耦合的地方。若科学目标是年周视差或自行，观测历元要覆盖视差因子的最大变化方向；若目标是 AGN 核位置，校准源结构和 core shift 需要进入误差预算；若目标是水脉泽，谱线通道中的 maser spot 位置必须和连续谱校准链保持一致。相位参考不是把相位“修好”的通用按钮，而是把相位稳定性、校准源几何和误差模型转化为可报告位置的测量方法。


![VLBI phase referencing astrometry](figures/practical_vlbi_phase_referencing_astrometry.png)

图中左上为校准源和目标源交替观测的时间结构，左下示意目标和校准源角距离越大，相位转移误差越容易增长。右侧把残余相位屏写成天球上的系统误差：目标和校准源看到的传播介质并不完全相同，转移后的相位残差会表现为位置偏差。


### 9.25.5 振幅定标、SEFD 与基线灵敏度

VLBI 的振幅定标通常比连通阵列更依赖先验台站信息。不同台站口径、接收机、效率、天气、仰角覆盖和记录系统差异很大，不能简单假设所有天线有相近灵敏度。常用的入口量是系统等效流量密度（System Equivalent Flux Density, SEFD）：

$$
\mathrm{SEFD}=\frac{2kT_{\rm sys}}{\eta_A A},
$$

其中 $T_{\rm sys}$ 是系统温度，$A$ 是几何口径面积，$\eta_A$ 是口径效率。SEFD 越低，台站越灵敏。真实定标还需要增益曲线、系统温度随时间和仰角的测量、采样与量化效率、天气和大气 opacity 修正，以及台站日志中的异常记录。

一条基线的热噪声可近似写为

$$
\sigma_{ij}\simeq \frac{\sqrt{\mathrm{SEFD}_i\mathrm{SEFD}_j}}{\eta_s\sqrt{2\Delta\nu\tau}},
$$

其中 $\Delta\nu$ 是有效带宽，$\tau$ 是积分时间，$\eta_s$ 是采样和相关效率。这个公式揭示了异质 VLBI 阵列的一个重要事实：一条弱基线由两个台站的 SEFD 几何平均决定，一个高 SEFD 台站会使相关基线的 fringe detection 更困难。成像阶段把许多基线合并可以改善热噪声，但 fringe fitting 阶段常要先在单条或少数组合基线上检测到相干信号，因此基线灵敏度直接影响数据能否进入后续校准。

振幅自校准在 VLBI 中也需要谨慎。若源模型不完整，振幅自校准可能把缺失结构或错误权重吸收到台站增益中；若完全放开绝对标尺，通量密度会偏离先验 SEFD 标定。实践中常先用 $T_{\rm sys}$、gain curve 和 opacity 建立 a priori 标尺，再用强而结构可控的源检查台站振幅一致性；只有当模型足够可靠、解区间足够长、闭合振幅残差有明确改善时，才把振幅自校准作为精细修正。


![VLBI amplitude SEFD baseline sensitivity](figures/practical_vlbi_amplitude_sefd_baseline_sensitivity.png)

左图示意不同台站的 SEFD 可以相差数十到数百倍。右图用同一带宽和积分时间估计异质阵列中的基线热噪声；低 SEFD 台站之间的基线最灵敏，高 SEFD 台站参与的基线更难 fringe detection。这个差异是 VLBI 先验振幅定标和 fringe 质量检查的核心背景。


### 9.25.6 VLBI 成像、动态范围与科学案例

VLBI 成像面对的是稀疏而很长的 $uv$ 覆盖。长基线提供高角分辨率，但缺少短基线意味着扩展结构会被过滤；台站数量少或观测时间短会带来高旁瓣 dirty beam；大气和钟差残差会限制相干动态范围。因此，VLBI 图像的质量常不由热噪声单独决定，而由残余相位、残余振幅、源模型不完整和 $uv$ 覆盖共同决定。

混合成图（hybrid mapping）是 VLBI 中常见策略：先用 fringe fitting 和相位参考建立可检测数据，再用 CLEAN 建立源模型，随后进行保守的相位自校准，必要时在模型稳定后加入振幅自校准。这个循环和第 9.4 节的自校准思想一致，但 VLBI 中模型偏差的代价更高。稀疏 $uv$ 覆盖容易让 CLEAN 组件填补旁瓣结构，过短解区间容易把噪声拟合成站点相位，过强自校准会移动源位置或压低扩展通量。因此，每一轮自校准后都要检查闭合相位、闭合振幅、残差图、分基线振幅、模型总通量和已知天体测量约束。

VLBI 选择的是高亮温、紧致或需要极高位置精度的科学问题。亮温常用近似式

$$
T_b\simeq 1.22\times10^{12}
\frac{S_\nu/\mathrm{Jy}}{(\nu/\mathrm{GHz})^2(\theta_{\rm maj}/\mathrm{mas})(\theta_{\rm min}/\mathrm{mas})}\ \mathrm{K}.
$$

这个式子说明，毫角秒尺度上仍有显著流量密度的源必然具有很高亮温，常对应 AGN 核、非热喷流、脉泽、致密爆发源或特殊瞬变。典型 VLBI 案例包括水脉泽年周视差和银河结构测量、AGN 喷流组件运动、ICRF 参考源位置、脉冲星精密天体测量、超新星和潮汐瓦解事件的紧致射电结构，以及毫米 VLBI 对黑洞阴影尺度结构的约束。


![VLBI imaging science cases](figures/practical_vlbi_imaging_science_cases.png)

图中把 VLBI 实践链和典型科学对象放在一起。稀疏长基线 $uv$ 覆盖提供极高分辨率，但也带来强旁瓣和模型依赖；紧致核、喷流组件和脉泽 spot 是 VLBI 最自然的目标。处理链从记录基带和相关开始，经过 fringe fitting、相位参考或自校准，最后进入成像、天体测量和物理解释。


### 9.25.7 从真实软件工作流看 VLBI 数据处理

VLBI 软件生态比普通连续谱成像更强调数据格式、相关器产品和台站先验信息。相关阶段常涉及 DiFX、SFXC 或专用相关器；传统厘米波 VLBI 处理中，AIPS 仍然常见，尤其用于 FITS-IDI 导入、先验振幅定标、FRING fringe fitting、CL 表管理和相位参考转移；CASA 也可承担部分 VLBI 校准、fringefit、成像和 Measurement Set 工作流；Difmap 常用于紧致源 hybrid mapping；HOPS 在 geodetic VLBI 和 EHT 相关处理传统中很重要；rPICARD、eht-imaging 等工具则体现了现代 pipeline 和闭合量建模的发展方向。

从教材角度看，软件名不应盖过数据对象。一个 VLBI 项目通常从 schedule、台站日志、ANTAB 或 $T_{\rm sys}$ 文件、gain curve、FITS-IDI 或 Measurement Set 开始；经过 a priori flag、量化修正、parallactic angle 或 feed 相关修正、ionosphere/troposphere 修正、fringe finder 延迟解、校准源相位解、目标源转移和成像；最后输出图像、模型、校准表、fringe 诊断、位置拟合表和误差预算。每一步都应说明改变的是可见度、权重、校准表、源模型、坐标参考还是科学解释。

| 阶段 | 主要对象 | 关键诊断 |
|---|---|---|
| 相关与导入 | 基带记录、FITS-IDI、Measurement Set | 台站时间、频率设置、延迟模型版本、缺失扫描 |
| 先验定标 | $T_{\rm sys}$、gain curve、opacity、flag | SEFD 合理性、坏台站、采样效率、振幅跳变 |
| Fringe fitting | fringe finder、相位参考源、delay-rate 解 | fringe S/N、delay/rate 峰、IF 一致性、参考天线稳定性 |
| 相位转移 | 校准表、目标源 visibility | 校准源结构、相位连接、目标检测、位置保持 |
| 成像与自校准 | CLEAN 模型、残差、闭合量 | 旁瓣、残差结构、闭合相位、通量守恒、动态范围 |
| 天体测量或物理解释 | 图像、spot/component 表、位置拟合 | 误差底、历元一致性、校准源系统误差、模型选择 |

这个表的作用不是替代软件手册，而是把命令和测量方程连接起来。VLBI 失败很少表现为单个命令报错，更多表现为 fringe 峰不稳定、某个台站相位跳变、相位参考后目标位置随频率漂移、振幅标尺在短基线和长基线之间不一致，或自校准后科学约束被悄悄吸收。诊断这些现象，需要同时看软件输出、台站先验信息和物理误差来源。


### 9.25.8 教学案例：22 GHz 水脉泽年周视差项目

一个适合教学展开的 VLBI 案例，是银河系恒星形成区中 $22\,\mathrm{GHz}$ 水脉泽的年周视差测量。科学目标是得到距离和自行，而不是仅仅得到一张高分辨率图像。观测设计要选择足够亮、足够紧致的 maser channel，同时选择距离目标尽可能近的 ICRF 校准源。由于 $22\,\mathrm{GHz}$ 对流层相位变化较快，校准源和目标源之间的切换周期需要足够短；多个历元要分布在年周视差因子变化最敏感的时段。

数据处理时，首先用强 fringe finder 检查全阵延迟、频带间相位和台站钟行为，再用 $T_{\rm sys}$ 与 gain curve 建立先验振幅标尺。随后在相位参考源上 fringe fitting，把相位、delay 和 rate 解转移到 maser 目标。目标 cube 中通常只在少数速度通道有强 maser spot，因此成像不是普通宽带连续谱成像，而是围绕窄通道 spot 的位置测量。每个历元中，spot 位置要相对于参考源坐标报告；跨历元拟合时，位置模型通常包含常数项、自行项和年周视差项。

这个案例能同时训练几个关键判断。若某个历元天气差，fringe S/N 和相位连接会先恶化；若校准源有结构，目标位置可能出现系统偏差；若 maser spot 本身结构变化，跨历元同名 spot 可能并非同一个物理团块；若只报告热噪声位置误差，距离误差会被低估。因此，最终结果应包含每个历元的图像、spot 拟合、相位参考诊断、校准源结构检查和误差底估计。这样的案例把 VLBI 从“高分辨率成像技术”推进到“带系统误差模型的精密测量”。


### 9.25.9 本节结论

VLBI 与普通连通阵列共享可见度和校准语言，但独立台站、独立时钟、长基线和稀疏 $uv$ 覆盖使延迟模型、fringe fitting、相位参考和 SEFD 定标成为核心环节。fringe fitting 解决的是残余相位、群延迟和 fringe rate 的相干检测问题；相位参考把校准源的相位稳定性转化为目标源相对位置；振幅定标依赖系统温度、增益曲线和异质基线灵敏度；成像和自校准必须始终受到闭合量、位置约束和误差预算的约束。

把 VLBI 放入第 9 章实践链后，可以形成一条自然延伸：普通数据检查和校准建立处理习惯，成像和自校准建立模型意识，观测设计和软件生态建立可追溯性，而 VLBI 进一步强调相干检测、精密时间、系统误差和天体测量。这个入口足以支撑后续扩展为独立 VLBI 专题，也能为 maser、AGN jet、脉冲星和高精度参考系案例提供共同基础。
